## Goal: - 

The goal here is to replicate a small GPT. The idea here is to train on a dataset so that the model starts to produce the next characters based on its learning. This based on all attention is all you need paper.

### Selecting the dataset 
Here the dataset selected the dataset is Tiny Shakespeare Dataset (All of shakespeare concantenated into one file)

In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("Length of dataset in characters:" , len(text))
print(text[:100])

Length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


### Preparing the dataset

We need to tokenize the words (basically convert the text to numbers for AI to understand)

In [4]:
## First lets build the vocabulary -> That means the total number of unique characters present in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print("Vocabulary Size:", vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary Size: 65


In [7]:
# Since this is a character level model we are tokenizing all characters

# This is a lookup table
stoi = { ch:i for i,ch in enumerate(chars)} # string to integer
itos = { i:ch for i,ch in enumerate(chars)} # integer to string
encode = lambda s: [stoi[c] for c in s] # encoder: take a string and output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

## So here we have set of vocab, which is our vocab
## Google uses sentencepiece tokenizer -> A different tokenization technique (sub word unit level)
## OpenAI uses -> Tiktoken (Byte Pair encoding) -> 50000 tokens(approx)

In [6]:
# Because of character level, we have long training sequences
print(encode("hi there"))
print(decode(encode("Hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
Hi there


### Preparing the dataset

In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # This is basically the dataset represented in numbers; the first 100 characters

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [ ]:
## Let's split the dataset into train, val and test -> Basic Splitting
TRAIN_SPLIT = 0.8
TRAIN_CUT = int(TRAIN_SPLIT * len(data))
print("Train Cut:", TRAIN_CUT)
TRAIN_DATA = data[:TRAIN_CUT]

VAL_SPLIT, TEST_SPLIT = 0.1, 0.1
VAL_CUT, TEST_CUT = int(VAL_SPLIT * len(data)), int(TEST_SPLIT * len(data))
print("Val/Test Cut:", TRAIN_CUT+VAL_CUT, TRAIN_CUT+VAL_CUT+TEST_CUT)
VAL_DATA = data[TRAIN_CUT: TRAIN_CUT+VAL_CUT]
TEST_DATA = data[TRAIN_CUT+VAL_CUT+TEST_CUT:]

print("Train first 10 char:", decode(TRAIN_DATA[:10].tolist()))
print("VAL first 10 char:", decode(VAL_DATA[:10].tolist())),
print("Test first 10 char:", decode(TEST_DATA[:10].tolist())),

Train Cut: 892315
Val/Test Cut: 1003854 1115393
Train first 10 char: First Citi
VAL first 10 char: you are,
T
Test first 10 char: 



(None,)

### Important note
We don't feed the entire data at once. When we are training such transformers, we randomly sample the chunk of such training set and train on them, of some fixed maximum length(block size/context length)

Why Chunking/Block size : - 
<ol>
<li>It gives the model many training examples from one long sequence.
<li>It limits memory and computation. Attention gets much more expensive as context length increases.
<li>It teaches next-token prediction at different context lengths.
<li>It defines how far back the model can look when making a prediction. It helps model learn from not just one character. 
</ol>

In [ ]:
block_size = 8
TRAIN_DATA[:block_size+ 1], decode(TRAIN_DATA[:block_size+1].tolist())

# THis is not just one example to train, here F predicts i, Fi predicts r.. 
# So these are 8 examples within itself

(tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]), 'First Cit')

In [48]:
x= TRAIN_DATA[:block_size]
y = TRAIN_DATA[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {decode(context.tolist())} OR {context} the target: {decode([target.item()])} OR {target}")

When input is F OR tensor([18]) the target: i OR 47
When input is Fi OR tensor([18, 47]) the target: r OR 56
When input is Fir OR tensor([18, 47, 56]) the target: s OR 57
When input is Firs OR tensor([18, 47, 56, 57]) the target: t OR 58
When input is First OR tensor([18, 47, 56, 57, 58]) the target:   OR 1
When input is First  OR tensor([18, 47, 56, 57, 58,  1]) the target: C OR 15
When input is First C OR tensor([18, 47, 56, 57, 58,  1, 15]) the target: i OR 47
When input is First Ci OR tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: t OR 58


In [57]:
# Now this is the time dimension of the transformer, Now let us focus on batch (efficiency for GPU's)

torch.manual_seed(1337)
batch_size = 4      # How many independent batches to process
block_size =8       ## Minimum context length


def get_batch(split):
    data = TRAIN_DATA if split=='train' else VAL_DATA
    ix =torch.randint(len(data) -block_size, (batch_size,))
    print("Batch cuts:", ix)
    x= torch.stack([data[i: i+block_size] for i in ix])
    y= torch.stack([data[i+1: i+block_size+1] for i in ix])
    return x, y
xb, yb = get_batch('train')

Batch cuts: tensor([188288,  12671, 712251, 226939])


In [73]:
print("Inputs:", xb.shape)
[(i, decode(i.tolist()))for i in xb]

Inputs: torch.Size([4, 8])


[(tensor([58, 63,  8,  0,  0, 19, 24, 27]), 'ty.\n\nGLO'),
 (tensor([39, 59, 45, 46, 58,  1, 46, 43]), 'aught he'),
 (tensor([49, 43, 57,  1, 53, 50, 42,  1]), 'kes old '),
 (tensor([52, 41, 47, 43, 52, 58,  1, 56]), 'ncient r')]

In [74]:
print("targets:", yb.shape)
[(i, decode(i.tolist()))for i in yb]

targets: torch.Size([4, 8])


[(tensor([63,  8,  0,  0, 19, 24, 27, 33]), 'y.\n\nGLOU'),
 (tensor([59, 45, 46, 58,  1, 46, 43,  1]), 'ught he '),
 (tensor([43, 57,  1, 53, 50, 42,  1, 46]), 'es old h'),
 (tensor([41, 47, 43, 52, 58,  1, 56, 47]), 'cient ri')]

In [ ]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        # print(context, target)
        print(f"When input is {decode(context.tolist())} OR {context.tolist()} the target: {decode([target.item()])} OR {target}")

When input is t OR [58] the target: y OR 63
When input is ty OR [58, 63] the target: . OR 8
When input is ty. OR [58, 63, 8] the target: 
 OR 0
When input is ty.
 OR [58, 63, 8, 0] the target: 
 OR 0
When input is ty.

 OR [58, 63, 8, 0, 0] the target: G OR 19
When input is ty.

G OR [58, 63, 8, 0, 0, 19] the target: L OR 24
When input is ty.

GL OR [58, 63, 8, 0, 0, 19, 24] the target: O OR 27
When input is ty.

GLO OR [58, 63, 8, 0, 0, 19, 24, 27] the target: U OR 33
When input is a OR [39] the target: u OR 59
When input is au OR [39, 59] the target: g OR 45
When input is aug OR [39, 59, 45] the target: h OR 46
When input is augh OR [39, 59, 45, 46] the target: t OR 58
When input is aught OR [39, 59, 45, 46, 58] the target:   OR 1
When input is aught  OR [39, 59, 45, 46, 58, 1] the target: h OR 46
When input is aught h OR [39, 59, 45, 46, 58, 1, 46] the target: e OR 43
When input is aught he OR [39, 59, 45, 46, 58, 1, 46, 43] the target:   OR 1
When input is k OR [49] the target: e O

### Basic Model Training
Now that we have prepared data, let us feed it to the model

In [82]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [83]:
print(xb.shape)

torch.Size([4, 8])


In [106]:
from typing import Any


class BiagramLanguageModel(nn.Module):
    def __init__(self, vocab_size, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        
        ## The table here is feature embedding or word embedding table. 
        ## Whenever we feed the batch of 2 dimension data [B,T] it does a lookup and then we get a (B,T,C(feature))
        self.token_embedding_lookup_table = nn.Embedding(vocab_size, vocab_size) ## (B, T, C)
        
    def forward(self,idx, targets=None):
        logits = self.token_embedding_lookup_table(idx)
        
        ## Cross entropy loss except input (B,C,T) -> MultiDimensional Input the channels should be second input
        if targets is None:
            loss = None
        else:
            ## here we reshape logits
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            ## We create it to 2D dimension
            targets = targets.view(-1) # (B, T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        ## idx here is the input, The (B,T) given here in the current context
        for _ in range(max_new_tokens):
            ### Get the Predictions
            logits, loss = self(idx)
            
            ## Focus on the last Step (output of last, input of next)
            logits = logits[:, -1, :]
            
            ## Apply softmax for probabilities
            probs = F.softmax(logits, dim=1) # Becomes (B,C)
            
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx =torch.cat([idx, idx_next], dim=1) #(B, T+1)
        return idx
        
model= BiagramLanguageModel(vocab_size)
logits, loss = model.forward(xb, yb)
print(logits.shape)
print(loss)
### Since are vocab is 65 in size, so our loss i -1*ln(1/65) -> Probable loss

print("Random Generation:",decode(model.generate(idx= torch.zeros((1,1), dtype =torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8893, grad_fn=<NllLossBackward0>)
Random Generation: 
uiK?HfyxOCoIr3sa-oTEWjX:itOkopzH&d$BC: QlOCDIEj?hrlWHVh3:j!dr3:XC3&s
cay.mesjpuqeLwa
;doIqR- !dHYo
v


#### Training this Model

In [ ]:
batch_size =32
